In [1]:
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import sys, os
from autograd import grad, hessian
import autograd.numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn import svm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
import pandas as pd
# from utils import *
from sklearn.metrics import mean_squared_error
import matplotlib.colors as mcolors

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

import joblib
from sklearn.model_selection import GridSearchCV, PredefinedSplit

import sklearn


In [3]:
#Load the data 

data = np.load('../../data_splits_splot22f_1008.npz')
X_train = data['X_train']
y_train = data['y_train'][:, :1]
X_val = data['X_val']
y_val = data['y_val'][:, :1]
X_test = data['X_test']
y_test = data['y_test'][:, :1]

# print("Number of class 0 ", len(y_train[y_train==0]) + len(y_val[y_val==0]) + len(y_test[y_test==0]))
# print("Number of class 1 ", len(y_train[y_train==1]) + len(y_val[y_val==1]) + len(y_test[y_test==1]))
# print("Number of class 2 ", len(y_train[y_train==2]) + len(y_val[y_val==2]) + len(y_test[y_test==2]))
# print("Number of class 3 ", len(y_train[y_train==3]) + len(y_val[y_val==3]) + len(y_test[y_test==3]))

#Standard normalize the training data and use the mean and std to normalize the testing data

knn_scaler = StandardScaler().fit(X_train)
X_train = knn_scaler.transform(X_train)
X_val = knn_scaler.transform(X_val)
X_test = knn_scaler.transform(X_test)


print("Training set ", np.shape(X_train), np.shape(y_train))
print("Validation set ", np.shape(X_val), np.shape(y_val))
print("Testing set ", np.shape(X_test), np.shape(y_test))


[[3.]
 [1.]
 [2.]
 ...
 [1.]
 [2.]
 [1.]]
Training set  (17739, 5) (17739, 1)
Validation set  (3801, 5) (3801, 1)
Testing set  (3802, 5) (3802, 1)


In [4]:
# Create a split indicator array
split_index = np.concatenate([
    np.full(len(X_train), -1),  # All train samples get -1
    np.zeros(len(X_val))         # All val samples get 0
])

# Combine train and validation sets
X_train_val = np.vstack([X_train, X_val])
y_train_val = np.hstack([y_train, y_val])

# Create the predefined split
ps = PredefinedSplit(test_fold=split_index)

# Do a gridsearch using kNN
estimator_KNN = KNeighborsClassifier(algorithm='auto')

parameters_KNN = {
    'n_neighbors': [ 3, 5, 7, 9, 11, 15, 20, 25],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']}
grid_search_KNN = GridSearchCV(
    estimator= estimator_KNN,
    param_grid=parameters_KNN,
    scoring = 'balanced_accuracy',
    cv = ps)

grid_search_KNN.fit(X_train_val, y_train_val)

# Save the best-performing model 
joblib.dump(grid_search_KNN.best_estimator_, 'best_models/knn_best_model.pkl')
joblib.dump(knn_scaler, 'best_models/knn_scaler.pkl')

# Also save best parameters for reference
results = {
    'best_params': grid_search_KNN.best_params_,
    'best_score': grid_search_KNN.best_score_,
    'cv_results': grid_search_KNN.cv_results_
}

print(grid_search_KNN.best_params_)
joblib.dump(results, 'models/knn_training_info.pkl')


{'metric': 'manhattan', 'n_neighbors': 3, 'weights': 'distance'}


['models/knn_training_info.pkl']

In [5]:
model = grid_search_KNN.best_estimator_

# Generate predictions
y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

print("Test Accuracy: {:.3f}%".format(accuracy_score(y_test, y_pred_test)*100))
print("Test Balanced Accuracy: {:.3f}%".format(balanced_accuracy_score(y_test, y_pred_test)*100))


# Save everything you'll need for plotting
np.savez('results/knn_results.npz',
         y_pred_train=y_pred_train,
         y_pred_val=y_pred_val,
         y_pred_test=y_pred_test,
)

Test Accuracy: 95.318%
Test Balanced Accuracy: 90.674%


In [ ]:
# unique_rows = unique_rows.T
import matplotlib.colors as mcolors
unique_rows = np.array([[1.0,1.0]])
for unique in unique_rows:
    Mass1 = unique[0]
    Mass2 = unique[1]
    # labels = ['log10(b[RSUN])', 'log10(v_inf[km/s])', str(Mass1), str(round(Mass2,2))]
    labels = ['log10(rp/(R1+R2))', 'log10(v_inf[km/s])', str(Mass1), str(round(Mass2,2))]
    N_knn_gridsearch_plotter(X_train_og, y_train, X_test_og ,y_test, best_model_knn, knn_scaler,feature_idx=(0, 1), fixed_values={2: Mass1, 3: Mass1/Mass2}, labels = labels)